In [1]:
import pandas as pd
import numpy as np
import torch as th
import os
from tqdm import tqdm

from frag_gnn.runner import load_config, init_dataset

In [2]:
tqdm.pandas()

In [3]:
os.chdir("/h/adamo/frag-gnn/")

In [4]:
proc_dp = "data/proc/nist20mona23v3"
spec_df = pd.read_pickle(f"{proc_dp}/spec_df.pkl")
mol_df = pd.read_pickle(f"{proc_dp}/mol_df.pkl")
ann_df = pd.read_pickle(f"{proc_dp}/ann_df.pkl")

In [5]:
def ann_mz_rel_diffs_fn(row):
    exact_mzs = np.array(row["ann_exact_mzs"], dtype=np.float32)
    peak_mzs = np.array(row["ann_peak_mzs"], dtype=np.float32)
    return np.abs(exact_mzs - peak_mzs) / np.maximum(peak_mzs, 200.)

mz_rel_diffs = ann_df.apply(ann_mz_rel_diffs_fn, axis=1)
max_mz_rel_diffs = mz_rel_diffs.apply(np.max)
ann_df.loc[:,"ann_max_mz_rel_diff"] = max_mz_rel_diffs

In [6]:
def ann_stats_fn(row):
    exact_mzs = np.array(row["ann_exact_mzs"], dtype=np.float32)
    peak_mzs = np.array(row["ann_peak_mzs"], dtype=np.float32)
    # diff_mzs = np.abs(exact_mzs - peak_mzs) / np.maximum(exact_mzs, 200.)
    diff_mzs = np.abs(peak_mzs.reshape(-1,1) - exact_mzs.reshape(1,-1)) / np.maximum(peak_mzs.reshape(1,-1), 200.)
    diff_mzs_diag = np.diag(diff_mzs)
    tolerances = [1e-5, 2e-5, 5e-5, 1e-4]
    results = {}
    for tol in tolerances:
        # check correct match
        match_mzs = diff_mzs < tol
        match_mzs_diag = diff_mzs_diag < tol
        match_mzs_nodiag = match_mzs.copy()
        np.fill_diagonal(match_mzs_nodiag, False)
        # print(match_mzs.shape, match_mzs_nodiag.shape)
        all_good_match = np.all(match_mzs_diag)
        mean_good_match = np.mean((match_mzs_diag).astype(float))
        any_bad_match = np.any(match_mzs_nodiag)
        mean_bad_match = np.sum(match_mzs_nodiag) / (match_mzs_nodiag.size - match_mzs_nodiag.shape[0])
        results[f"ann_good_match_all_{tol}"] = all_good_match
        results[f"ann_good_match_mean_{tol}"] = mean_good_match
        results[f"ann_bad_match_any_{tol}"] = any_bad_match
        results[f"ann_bad_match_mean_{tol}"] = mean_bad_match
    return pd.Series(results)

ann_stats_df = ann_df.progress_apply(ann_stats_fn, axis=1)

In [7]:
print(ann_stats_df.mean())

In [8]:
m_spec_df = spec_df[["group_id","peaks"]].groupby("group_id").agg(list)

In [16]:
def merge_stats_func(*peakses):

    mzs = []
    for peaks in peakses:
        for peak in peaks:
            mzs.append(peak[0])
    mzs = np.array(mzs)
    mzs_r = np.round(mzs, 4)
    mzs_unique, mzs_counts = np.unique(mzs, return_counts=True)
    mzs_r_unique, mzs_r_counts = np.unique(mzs_r, return_counts=True)
    mzs_counts_mean = np.mean(mzs_counts)
    mzs_r_counts_mean = np.mean(mzs_r_counts)
    mzs_counts_max = np.max(mzs_counts)
    mzs_r_counts_max = np.max(mzs_r_counts)
    mzs_unmerged_total = mzs.shape[0]
    mzs_merged_total = mzs_unique.shape[0]
    mzs_r_merged_total = mzs_r_unique.shape[0]
    spectrum_total = len(peakses)
    results = pd.Series(
        {
            "mzs_counts_mean": mzs_counts_mean,
            "mzs_r_counts_mean": mzs_r_counts_mean,
            "mzs_counts_max": mzs_counts_max,
            "mzs_r_counts_max": mzs_r_counts_max,
            "mzs_unmerged_total": mzs_unmerged_total,
            "mzs_merged_total": mzs_merged_total,
            "mzs_r_merged_total": mzs_r_merged_total,
            "spectrum_total": spectrum_total
        }
    )
    return results

In [ ]:
m_spec_stats = m_spec_df["peaks"].progress_apply(lambda x: merge_stats_func(*x))

In [ ]:
m_spec_stats.mean()

In [ ]:
m_spec_stats.shape

In [5]:
split_dp = "data/split/nist20mona23v2_d4_wr00_inchikey"
train_ids_df_1 = pd.read_csv(f"{split_dp}/train_ids.csv")
val_ids_df_1 = pd.read_csv(f"{split_dp}/val_ids.csv")
test_ids_df_1 = pd.read_csv(f"{split_dp}/test_ids.csv")
# all_ids = np.concatenate([train_ids,val_ids,test_ids],axis=0)

In [6]:
template_fp = "config/template.yml"
custom_fp = "config/exp_graff/max-iso-2.yml"
config_d = load_config(template_fp, custom_fp)

In [7]:
# train_ds, val_ds, test_ds = init_dataset(config_d, splits=("train","val","test"))

In [8]:
# Check for both unmerged and merged:
# 1. (DONE) fraction of spectra with any annotations
# 2. fraction with mismatch annotations
# 3. (DONE) fraction with annotations and molecular weight < 1000 Da
# 4. (DONE) fraction or annotated spectra with normalized collision energy in {20, 35, 50}

In [9]:
def calculate_stats(split_ids_df, spec_df, mol_df, ann_df):

    split_spec_df = spec_df[spec_df["group_id"].isin(split_ids_df["group_id"])]
    split_mol_df = mol_df[mol_df["mol_id"].isin(split_ids_df["mol_id"])]
    split_ann_df = ann_df[ann_df["spec_id"].isin(split_spec_df["spec_id"])]
    split_spec_mol_df = split_spec_df.merge(split_mol_df, on="mol_id", how="inner")
    split_ann_mask = (split_spec_df["spec_id"].isin(ann_df["spec_id"])).to_numpy()
    split_ann_frac = split_ann_mask.mean()
    print(f"> Fraction of spectra with any annotations = {split_ann_frac}")
    split_light_mask = (split_spec_mol_df["exact_mw"] <= 1000.).to_numpy()
    split_ann_light_mask = split_ann_mask & split_light_mask
    split_light_frac = split_light_mask.mean()
    split_ann_light_frac = split_ann_light_mask.mean()
    print(f"> Fraction of spectra with molecular weight <= 1000 Da = {split_light_frac}")
    print(f"> Fraction of annotated spectra with molecular weight <= 1000 Da = {split_ann_light_frac}")
    split_prec_light_mask = (split_spec_mol_df["prec_mz"] <= 1000.).to_numpy()
    split_ann_prec_light_mask = split_ann_mask & split_prec_light_mask
    split_prec_light_frac = split_prec_light_mask.mean()
    split_ann_light_frac = split_ann_prec_light_mask.mean()
    print(f"> Fraction of spectra with precursor m/z <= 1000 Da = {split_prec_light_frac}")
    print(f"> Fraction of annotated spectra with precursor m/z <= 1000 Da = {split_ann_light_frac}")
    split_ce_mask = ((split_spec_df["nce"] == 20) | (split_spec_df["nce"] == 35) | (split_spec_df["nce"] == 50)).to_numpy()
    split_ann_ce_mask = split_ann_mask & split_ce_mask
    split_ce_frac = split_ce_mask.mean()
    split_ann_ce_frac = split_ann_ce_mask.mean()
    print(f"> Fraction of spectra with NCE in {{20, 35, 50}} = {split_ce_frac}")
    print(f"> Fraction of annotated spectra with NCE in {{20, 35, 50}} = {split_ann_ce_frac}")
    for t in [1e-5, 2e-5, 3e-5, 5e-5, 1e-4]:
        split_tol_mask = split_ann_df["ann_max_mz_rel_diff"] <= t
        split_tol_frac = split_tol_mask.mean()
        print(f"> Fraction of annotated spectra with relative peak mz differences <= {t} = {split_tol_frac}")    


In [ ]:
split_to_split_ids_df = {
    "train": train_ids_df_1,
    "val": val_ids_df_1,
    "test": test_ids_df_1
}

for split, split_ids_df in split_to_split_ids_df.items():
    print(f">> Split = {split}")
    calculate_stats(split_ids_df, spec_df, mol_df, ann_df)

In [11]:
# measure performance on spectra without annotations
# measure performance on spectra with only (20, 35, 50) collision energies

In [12]:
graff_output_fp = "inference_outputs/v3_feb5/inchikey/graff/s0/test.pkl"
graff_output_d = th.load(graff_output_fp)

In [ ]:
graff_output_d.keys()

In [ ]:
graff_output_d["cos_sim_0.01"].mean()

In [15]:
graff_ids = graff_output_d["unique_id"].numpy()
graff_cos_sim = graff_output_d["cos_sim_0.01"].numpy()

In [ ]:
test_ann_ids = test_ids_df_1[test_ids_df_1["spec_id"].isin(ann_df["spec_id"]).to_numpy()]["group_id"].to_numpy()
print(len(test_ann_ids), len(test_ids_df_1))

In [ ]:
print(graff_cos_sim.mean())
print(graff_cos_sim[np.isin(graff_ids,test_ann_ids)].mean())
print(graff_cos_sim[~np.isin(graff_ids,test_ann_ids)].mean())